In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.evaluation.backtest import VARIABLE_SERIES

meta = json.loads((ROOT / "models" / "model_meta.json").read_text(encoding="utf-8"))
res = pd.read_parquet(ROOT / "data" / "processed" / "backtest_predictions_v2.parquet")
users = (
    pd.read_csv(ROOT / "data" / "raw" / "synthetic_transactions.csv",
                usecols=["user_id", "archetype"])
    .drop_duplicates("user_id")
)
res = res.merge(users, on="user_id", how="left")

print("Champion:", meta["champion"], "| version:", meta["model_version"])
print({k: round(v, 3) for k, v in meta["backtest_metrics"].items()})

Champion: lgbm | version: lgbm-2026-10-25
{'variable_mae': 53.547, 'baseline_mae': 65.564, 'threshold': 59.008, 'coverage_80': 0.824, 'total_mae_sum': 262.813}


In [2]:
def mae(g, col):
    return (g["target"] - g[col]).abs().mean()

by_series = pd.DataFrame({
    category: {
        "model": mae(g, "pred"),
        "last_month": mae(g, "lag_1"),
        "same_month_last_year": mae(g, "lag_12"),
    }
    for category, g in res.groupby("category")
}).T.round(2)
best_baseline = by_series[["last_month", "same_month_last_year"]].min(axis=1)
by_series["better_than_best_baseline_%"] = ((1 - by_series["model"] / best_baseline) * 100).round(1)
by_series

,model,last_month,same_month_last_year,better_than_best_baseline_%
Dining,39.80,54.61,47.25,15.8
Groceries,28.58,33.55,34.62,14.8
Health,15.79,26.86,25.71,38.6
Income,268.94,464.77,299.22,10.1
Leisure,157.08,229.50,170.13,7.7
Other,6.74,50.60,8.37,19.5
Rent,0.00,0.00,0.00,NaN
Shopping,76.92,120.99,94.24,18.4
Subscriptions,0.00,0.00,0.00,NaN
Total expenses,274.20,412.89,308.65,11.2


In [3]:
var = res[res["category"].isin(VARIABLE_SERIES)]
income = res[res["category"] == "Income"]

def by_type(df):
    return pd.DataFrame({
        a: {"model": mae(g, "pred"), "same_month_last_year": mae(g, "lag_12")}
        for a, g in df.groupby("archetype")
    }).T.round(2)

print("Variable spending, MAE by user type")
display(by_type(var))
print("Income, MAE by user type")
display(by_type(income))

Variable spending, MAE by user type


,model,same_month_last_year
freelancer,51.50,64.70
salaried_family,77.20,88.70
salaried_single,46.68,59.55
student,28.85,37.16


Income, MAE by user type


,model,same_month_last_year
freelancer,846.93,1051.05
salaried_family,137.27,70.74
salaried_single,60.62,56.16
student,16.69,7.66


# Model experiments: findings

**Champion:** global LightGBM (median plus 10% and 90% quantile models), with Rent and
Subscriptions copied from last month. Backtest: 12 one-month-ahead folds, 2025-10 to 2026-09.

| | Result | Target |
|---|---|---|
| Variable spending MAE | 53.55 vs 65.56 for the best baseline (same month last year), 18% lower | at least 10% |
| Cold start (60 users never seen in training) | 52.70 vs 62.81, 16% lower | at least 10% |
| 80% interval coverage | 0.824 seen users, 0.835 cold start | 0.75 to 0.85 |
| Total expenses MAE | 262.81 from summing categories, 274.20 forecasting the total directly, 308.65 best baseline | |

**Where it wins:** Dining, Groceries, Health, Shopping, Transport and Other, clearly. Income
about 10% better than the best baseline, but it has the largest absolute error.
Rent and Subscriptions are fixed amounts and are copied exactly.

**Where it does not win:**
- Utilities: the model scores 9.91, copying the same month last year scores 8.25.
- Leisure: only about 8% better than the best baseline.
- Income by user type: (add the numbers from cell 3 here, especially freelancers).

**Caveats:**
- All users are synthetic. The model learned the generator's patterns (yearly seasonality,
  December and summer spikes), not real behaviour. Real data may rank the baselines differently.
- The categoriser scores 100% on unseen users but the data has only 95 unique descriptions.
  On descriptions it had never seen, accuracy was 40.8%, so the keyword rules matter.
- Macro data ends in December 2025, so 9 of the 12 test months have no new macro values.
- Anomaly model: evaluated only on anomalies I injected (duplicates, spikes, unknown merchants).
  The simple rules find every duplicate and about 65% of spikes; the forest only helps with
  unknown merchants and is noisy.

In [4]:
inside = (var["target"] >= var["lo"]) & (var["target"] <= var["hi"])
inside.groupby(var["archetype"]).mean().round(3)

archetype
freelancer         0.837
salaried_family    0.799
salaried_single    0.834
student            0.822
dtype: float64